In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Replicator-Documentation Evaluator

This notebook evaluates whether the replicator's documentation faithfully reproduces the results and conclusions of the original experiment.

## Task Overview
- Compare original `documentation.md` with replicated `documentation_replication.md`
- Evaluate: DE1 (Result Fidelity), DE2 (Conclusion Consistency), DE3 (No External Information)
- Output evaluation results to `evaluation/replication_eval/`

In [2]:
# Check for GPU availability
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU available, using CPU")

GPU available: NVIDIA A100 80GB PCIe
CUDA version: 12.4


In [3]:
# Define paths
ORIGINAL_REPO = "/net/scratch2/smallyan/filter_eval"
REPLICATION_DIR = "/net/scratch2/smallyan/filter_eval/evaluation/replications"
OUTPUT_DIR = "/net/scratch2/smallyan/filter_eval/evaluation/replication_eval"

# Check if paths exist
print(f"Original repo exists: {os.path.exists(ORIGINAL_REPO)}")
print(f"Replication dir exists: {os.path.exists(REPLICATION_DIR)}")

# List contents of original repo
print("\n--- Original repo contents ---")
if os.path.exists(ORIGINAL_REPO):
    for item in sorted(os.listdir(ORIGINAL_REPO)):
        print(f"  {item}")

Original repo exists: True
Replication dir exists: True

--- Original repo contents ---
  .git
  .gitignore
  .remote_jobs
  CodeWalkthrough.md
  conda_env.yml
  data
  data_save
  demo.ipynb
  documentation.pdf
  env.yml
  env_demo.yml
  evaluation
  globals.yml
  hparams
  notebooks
  plan.md
  results
  run_finetuning.py
  run_jobs
  run_monitor.py
  scripts
  src
  test_suite


In [4]:
# Look for documentation.md in original repo
import glob

# Search for documentation files in original repo
doc_patterns = [
    f"{ORIGINAL_REPO}/documentation.md",
    f"{ORIGINAL_REPO}/**/documentation.md",
    f"{ORIGINAL_REPO}/README.md",
    f"{ORIGINAL_REPO}/**/*.md"
]

print("Searching for markdown documentation in original repo:")
md_files = glob.glob(f"{ORIGINAL_REPO}/**/*.md", recursive=True)
for f in sorted(md_files):
    print(f"  {f}")

Searching for markdown documentation in original repo:
  /net/scratch2/smallyan/filter_eval/CodeWalkthrough.md
  /net/scratch2/smallyan/filter_eval/evaluation/replications/documentation_replication.md
  /net/scratch2/smallyan/filter_eval/evaluation/replications/evaluation_replication.md
  /net/scratch2/smallyan/filter_eval/plan.md
  /net/scratch2/smallyan/filter_eval/src/rome/README.md


In [5]:
# Check for PDF documentation
print("PDF files in original repo:")
pdf_files = glob.glob(f"{ORIGINAL_REPO}/**/*.pdf", recursive=True)
for f in pdf_files:
    print(f"  {f}")

# List replication directory contents
print("\n--- Replication directory contents ---")
if os.path.exists(REPLICATION_DIR):
    for item in sorted(os.listdir(REPLICATION_DIR)):
        print(f"  {item}")

PDF files in original repo:
  /net/scratch2/smallyan/filter_eval/documentation.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/causality_diff_approach.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/gemma-head_transfer.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/lamb_vs_causality_and_heads.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/causality_vs_n_distractors.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/head_level_transfer.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/pred_transfer.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/fig_1-crop.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/probe_performance.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/causality_confusion_matrix.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/head_transfer.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/figures/avg_trick_scores.pdf
  /net/scratch2/smallyan/filter_eval/notebooks/f

In [6]:
# Read the original documentation PDF
import fitz  # PyMuPDF

pdf_path = f"{ORIGINAL_REPO}/documentation.pdf"
doc = fitz.open(pdf_path)

original_doc_text = ""
for page_num in range(len(doc)):
    page = doc.load_page(page_num)
    original_doc_text += page.get_text()

doc.close()

print(f"Original documentation length: {len(original_doc_text)} characters")
print("\n--- Original Documentation Content ---\n")
print(original_doc_text[:10000])  # First 10000 chars

Original documentation length: 95732 characters

--- Original Documentation Content ---

Under Review
LLMS PROCESS LISTS WITH GENERAL FILTER HEADS
Arnab Sen Sharma∗, Giordano Rogers, Natalie Shapira, and David Bau
Khoury College of Computer Sciences, Northeastern University
ABSTRACT
We investigate the mechanisms underlying a range of list-processing tasks in LLMs,
and we find that LLMs have learned to encode a compact, causal representation of
a general filtering operation that mirrors the generic “filter” function of functional
programming. Using causal mediation analysis on a diverse set of list-processing
tasks, we find that a small number of attention heads, which we dub filter heads,
encode a compact representation of the filtering predicate in their query states at
certain tokens. We demonstrate that this predicate representation is general and
portable: it can be extracted and reapplied to execute the same filtering operation
on different collections, presented in different form

In [7]:
# Print more of the original documentation
print(original_doc_text[10000:20000])

uery state for
[ℓ, j] at the last token position, qℓj
−1, hereafter denoted as qsrc for brevity.
destination run
M(pdest):
The LM is run with pdest.
patched run
M(pdest)[←qsrc]:
We run the LM with pdest again, but we replace the query state
at the last token position for head [ℓ, j], qℓj
−1 with qsrc cached from the source run.
The attention patterns for the head [ℓ, j] from the three forward passes for an example prompt pair are
depicted in Figure 1(b), (e), and (f) respectively. In the source and destination runs, the head attends
to the items that satisfy the respective predicates. But in the patched run, the filter head [ℓ, j] shifts its
attention to the item in Cdest that satisfies ψsrc. Patching qsrc is enough to trigger the execution of
ψsrc for this head in a different context, validating that qsrc encodes a compact representation of ψsrc.
Notably, we cache the query states before the positional embedding (Su et al., 2024) is applied, while
Attn(qt, K) in Equation (1) is calcul

In [8]:
# Print more of the original documentation - results section
print(original_doc_text[20000:35000])

on specific linguistic forms. However, we
also observe that when the question is presented before the items, the filter heads show poor causality
(see Table 2(c)). We find that this is because in the question-before case the LM relies more on a
complementary implementation of filtering, which we discuss in Section 5 and in Section B. All the
other results presented in this section are calculated on prompts following the question-after format.
1This slightly increases the causality by removing the order information. See Section F.
5
Under Review
Table 2: Portability of predicate representations across linguistic variations. The predicate vector qsrc is
extracted from a source prompt and patched to destination prompts in (a) different languages, (b) different
presentation formats for the items, and (c) placing the question before or after presenting the collection.
To
From
English
Spanish
French
Hindi
Thai
English
0.863
0.893
0.779
0.928
0.951
Spanish
0.857
0.877
0.775
0.875
0.891
French

In [9]:
# Print conclusion section of the original documentation
print(original_doc_text[35000:50000])

 al. (2024) have found function vector heads that encode task representations
that are transportable across contexts. Filter heads are an addition to this class of attention heads that
show distinct functional specialization.
LM Selection Mechanisms.
A few empirical studies have explored the selection mechanism in
LMs, primarily in MCQA settings. Tulchinskii et al. (2024) identifies “select-and-copy” heads based
on their attention pattern that focus on “\n” after a correct item in a question-first MCQ format.
Lieberum et al. (2023) also identify attention heads that attend to the correct MCQ label/letter and
show that these “correct label” heads encode the ordering ID of the presented options. Wiegreffe et al.
(2025) showed that attention modules in the middle layers promote the answer symbols in a MCQA
task. Unlike these works focused on MCQA settings, in this paper we investigate list-processing
in general and find a set of filter heads that implement predicate evaluation that genera

In [10]:
# Print the rest of the conclusions and appendix sections
print(original_doc_text[50000:65000])

ssing systems,
36:27223–27250, 2023.
13
Under Review
A
EXAMPLE PROMPTS FROM OUR DATASET
A.1
DIFFERENT TASKS
Select One – Type of Object
“Options: Bus, Peach, Scooter, Phone, Pen
Find the fruit in the options presented
above.
Answer:”
Expected LM Output: “ Peach”
Select One – Type of profession
“Options: Neymar, Hillary Clinton, Clint
Eastwood
Who among these people mentioned above is
an actor by profession?
Answer:”
Expected LM Output: “ Clint”
Select One – Type of nationality
“Options: Ronaldinho, Brad Pitt, Jet Li,
Ken Watanabe
Who among these people mentioned above is
from China?
Answer:”
Expected LM Output: “ Jet”
Select One – Location of landmark
“Options:
Cabo San Lucas Arch, Plaza de
Armas Cusco, Mont Saint-Michel
Which of these landmarks is in Peru?
Answer:”
Expected LM Output: “ Plaza”
Select One — Rhyme
“Options: blue, debt, bright, sting, sake
Which of these words rhymes with glue?
Answer:”
Expected LM Output: “ blue”
Select One (MCQ)
“a. Banana
b. Paperclip
c. Oven
d. Dress

In [11]:
# Read the replicated documentation
replication_path = f"{REPLICATION_DIR}/documentation_replication.md"
with open(replication_path, 'r') as f:
    replicated_doc = f.read()

print(f"Replicated documentation length: {len(replicated_doc)} characters")
print("\n--- Replicated Documentation Content ---\n")
print(replicated_doc)

Replicated documentation length: 3983 characters

--- Replicated Documentation Content ---

# Documentation: Replication of "LLMs Process Lists With General Filter Heads"

## Goal

Replicate the core experiment from the paper investigating how Large Language Models (LLMs) perform filtering operations over lists of items using specialized attention heads called "filter heads." The key hypothesis is that a small number of attention heads encode a compact representation of the filtering predicate in their query states, and this representation can be transferred between different contexts.

## Data

### Source Data
- **Dataset**: SelectOne task from the `data_save/selection/objects.json` file
- **Categories**: 16 object categories including fruit, vehicle, furniture, animal, etc.
- **Format**: Each category contains a list of objects (e.g., fruits: Apple, Banana, Orange, etc.)
- **Prompt Templates**: 4 templates available for constructing SelectOne prompts

### Sample Structure
Each sample

## Documentation Comparison

Now I will compare the original documentation with the replicated documentation to evaluate:
1. **DE1 - Result Fidelity**: Do the results match within tolerance?
2. **DE2 - Conclusion Consistency**: Are conclusions consistent?
3. **DE3 - No External Information**: Is there any hallucinated or external info?

In [12]:
# Extract key findings from original documentation for comparison
original_key_findings = """
ORIGINAL DOCUMENTATION KEY FINDINGS:

1. CORE CLAIM - Filter Heads:
   - LLMs have specialized attention heads called "filter heads" that encode filtering predicates
   - These heads are concentrated in the MIDDLE LAYERS (layers 30-55 in 80-layer Llama-70B)
   - They encode predicates as query states at specific tokens (typically last token position)
   
2. MAIN RESULTS (Llama-70B):
   - Causality score for Object Type filtering: 0.863
   - Causality score for Person Profession: 0.836
   - ΔLogit: +9.03 for Object Type
   - Filter heads retain causality with up to 7 distractors (>0.8)
   - Cross-lingual transfer shows high causality (0.775-0.957)
   
3. KEY TABLES/FIGURES:
   - Table 1: Causality of filter heads on SelectOne tasks
   - Table 2: Portability across linguistic variations
   - Table 3: LM performance drops 22.5-89.8% when filter heads ablated
   - Table 4: Filter heads are distinct from other head types
   - Figure 3: Transfer across different tasks (causality 0.36-0.90)
   
4. CONCLUSIONS:
   - Filter heads are localized in middle layers
   - Predicate representations are portable across formats, languages, tasks
   - Dual filtering mechanism: lazy (filter heads) vs eager (is_match flags)
   - Filter heads form a foundational layer for a range of reduce operations
"""
print(original_key_findings)


ORIGINAL DOCUMENTATION KEY FINDINGS:

1. CORE CLAIM - Filter Heads:
   - LLMs have specialized attention heads called "filter heads" that encode filtering predicates
   - These heads are concentrated in the MIDDLE LAYERS (layers 30-55 in 80-layer Llama-70B)
   - They encode predicates as query states at specific tokens (typically last token position)
   
2. MAIN RESULTS (Llama-70B):
   - Causality score for Object Type filtering: 0.863
   - Causality score for Person Profession: 0.836
   - ΔLogit: +9.03 for Object Type
   - Filter heads retain causality with up to 7 distractors (>0.8)
   - Cross-lingual transfer shows high causality (0.775-0.957)
   
3. KEY TABLES/FIGURES:
   - Table 1: Causality of filter heads on SelectOne tasks
   - Table 2: Portability across linguistic variations
   - Table 3: LM performance drops 22.5-89.8% when filter heads ablated
   - Table 4: Filter heads are distinct from other head types
   - Figure 3: Transfer across different tasks (causality 0.36-0.90)


In [13]:
# Extract key findings from replicated documentation
replicated_key_findings = """
REPLICATED DOCUMENTATION KEY FINDINGS:

1. CORE CLAIM - Filter Heads:
   - Correctly describes filter heads as encoding filtering predicates
   - Correctly describes query state caching and patching methodology
   
2. MAIN RESULTS (GPT-2):
   - Top heads by effect: Layer 11 Head 4 (Δ=0.6875), Layer 10 Head 7 (Δ=0.4375)
   - Mean Δ Track Object: 0.90 ± 1.43
   - Mean Δ Dest Object: 1.41 ± 1.22
   - Positive effect on track object: 60% of samples
   - Task Accuracy: <20%
   
3. COMPARISON TABLE:
   | Metric              | Original (Llama-70B) | Replication (GPT-2) |
   |---------------------|---------------------|---------------------|
   | Causality Score     | 0.836-0.863         | N/A                 |
   | ΔLogit              | 8.26 ± 3.35         | 0.90 ± 1.43         |
   | Task Accuracy       | ~100%               | <20%                |
   
4. CONCLUSIONS:
   - Methodology was replicated correctly
   - Effect magnitude much smaller with GPT-2
   - Cannot identify clear "filter heads" in GPT-2
   - Differences attributed to model size, training, architecture
"""
print(replicated_key_findings)


REPLICATED DOCUMENTATION KEY FINDINGS:

1. CORE CLAIM - Filter Heads:
   - Correctly describes filter heads as encoding filtering predicates
   - Correctly describes query state caching and patching methodology
   
2. MAIN RESULTS (GPT-2):
   - Top heads by effect: Layer 11 Head 4 (Δ=0.6875), Layer 10 Head 7 (Δ=0.4375)
   - Mean Δ Track Object: 0.90 ± 1.43
   - Mean Δ Dest Object: 1.41 ± 1.22
   - Positive effect on track object: 60% of samples
   - Task Accuracy: <20%
   
3. COMPARISON TABLE:
   | Metric              | Original (Llama-70B) | Replication (GPT-2) |
   |---------------------|---------------------|---------------------|
   | Causality Score     | 0.836-0.863         | N/A                 |
   | ΔLogit              | 8.26 ± 3.35         | 0.90 ± 1.43         |
   | Task Accuracy       | ~100%               | <20%                |
   
4. CONCLUSIONS:
   - Methodology was replicated correctly
   - Effect magnitude much smaller with GPT-2
   - Cannot identify clear "filter h

In [14]:
# DE1: Result Fidelity Evaluation
de1_analysis = """
=== DE1: RESULT FIDELITY EVALUATION ===

COMPARISON OF REPORTED RESULTS:

Original Documentation (Llama-3.3-70B-Instruct):
- Causality Score: 0.836-0.863 for Object Type/Profession
- ΔLogit: +9.03 for Object Type, +7.33 for Profession
- Task Accuracy: ~100% (model correctly predicts answers)
- Filter heads concentrated in middle layers (30-55 of 80)

Replicated Documentation (GPT-2):
- Causality Score: N/A (model cannot solve task reliably)
- ΔLogit: 0.90 ± 1.43 (approximately 10x smaller)
- Task Accuracy: <20%
- Top effect heads: Layers 1-11 (out of 12)

CRITICAL DIFFERENCES:
1. Model mismatch: Replication uses GPT-2 (124M) instead of Llama-3.3-70B (70B)
   - This is a 560x difference in model size
   - Different architecture (GPT-2 vs Llama)
   - GPT-2 is not instruction-tuned

2. Results do NOT match:
   - Causality cannot be computed due to task failure
   - ΔLogit is ~10x smaller
   - Task accuracy is ~5x worse
   - Cannot identify clear filter heads

3. Replication explicitly acknowledges these differences in comparison table

VERDICT: The replicated results do NOT match the original results. While the 
replication correctly reports what was observed with GPT-2, these results 
fundamentally differ from the original Llama-70B results. The replication 
failed to reproduce the core findings because the wrong model was used.

DE1 RESULT: FAIL
"""
print(de1_analysis)


=== DE1: RESULT FIDELITY EVALUATION ===

COMPARISON OF REPORTED RESULTS:

Original Documentation (Llama-3.3-70B-Instruct):
- Causality Score: 0.836-0.863 for Object Type/Profession
- ΔLogit: +9.03 for Object Type, +7.33 for Profession
- Task Accuracy: ~100% (model correctly predicts answers)
- Filter heads concentrated in middle layers (30-55 of 80)

Replicated Documentation (GPT-2):
- Causality Score: N/A (model cannot solve task reliably)
- ΔLogit: 0.90 ± 1.43 (approximately 10x smaller)
- Task Accuracy: <20%
- Top effect heads: Layers 1-11 (out of 12)

CRITICAL DIFFERENCES:
1. Model mismatch: Replication uses GPT-2 (124M) instead of Llama-3.3-70B (70B)
   - This is a 560x difference in model size
   - Different architecture (GPT-2 vs Llama)
   - GPT-2 is not instruction-tuned

2. Results do NOT match:
   - Causality cannot be computed due to task failure
   - ΔLogit is ~10x smaller
   - Task accuracy is ~5x worse
   - Cannot identify clear filter heads

3. Replication explicitly ac

In [15]:
# DE2: Conclusion Consistency Evaluation
de2_analysis = """
=== DE2: CONCLUSION CONSISTENCY EVALUATION ===

ORIGINAL DOCUMENTATION CONCLUSIONS:
1. Filter heads are specialized attention heads that implement filtering operations
2. Filter heads are localized in middle layers of transformer LMs
3. Predicate representations are portable across formats, languages, tasks
4. Dual filtering mechanism exists: lazy (filter heads) vs eager (is_match flags)
5. Filter heads form a foundational layer for reduce operations
6. Predicate encoding in query states can be extracted and reapplied

REPLICATED DOCUMENTATION CONCLUSIONS:
1. "Methodology": The core patching mechanism works correctly
2. "Head-Specific Effects": Some heads show stronger effects than others
3. "Effect Direction": Patching tends to increase track object logits

What Was NOT Replicated (explicitly stated):
1. "Effect Magnitude": Much smaller effects with GPT-2
2. "Task Performance": GPT-2 cannot reliably solve the SelectOne task
3. "Filter Head Localization": Cannot identify clear "filter heads" in GPT-2

CONCLUSION CONSISTENCY ANALYSIS:
- The replication does NOT confirm the original conclusions
- It explicitly states that filter head localization could NOT be replicated
- The core claim (filter heads exist and are identifiable) is NOT supported by replication
- The portable predicate representation claim cannot be validated (task not solved)

However, the replication documentation:
- Correctly attributes differences to model choice (GPT-2 vs Llama-70B)
- Does not make claims that contradict the original (just fails to replicate)
- Acknowledges limitations honestly

VERDICT: The conclusions differ significantly. The replication cannot confirm 
any of the original paper's key claims due to using a fundamentally different
and inadequate model. While it doesn't actively contradict the original, it 
fails to reproduce the core conclusions.

DE2 RESULT: FAIL
"""
print(de2_analysis)


=== DE2: CONCLUSION CONSISTENCY EVALUATION ===

ORIGINAL DOCUMENTATION CONCLUSIONS:
1. Filter heads are specialized attention heads that implement filtering operations
2. Filter heads are localized in middle layers of transformer LMs
3. Predicate representations are portable across formats, languages, tasks
4. Dual filtering mechanism exists: lazy (filter heads) vs eager (is_match flags)
5. Filter heads form a foundational layer for reduce operations
6. Predicate encoding in query states can be extracted and reapplied

REPLICATED DOCUMENTATION CONCLUSIONS:
1. "Methodology": The core patching mechanism works correctly
2. "Head-Specific Effects": Some heads show stronger effects than others
3. "Effect Direction": Patching tends to increase track object logits

What Was NOT Replicated (explicitly stated):
1. "Effect Magnitude": Much smaller effects with GPT-2
2. "Task Performance": GPT-2 cannot reliably solve the SelectOne task
3. "Filter Head Localization": Cannot identify clear "filter

In [16]:
# DE3: No External or Hallucinated Information Evaluation
de3_analysis = """
=== DE3: NO EXTERNAL OR HALLUCINATED INFORMATION ===

CHECKING FOR EXTERNAL/HALLUCINATED CONTENT IN REPLICATION:

1. METHODOLOGY DESCRIPTION:
   - Query state caching: Matches original (Section 2.2)
   - Query state patching: Matches original methodology
   - Effect measurement: Consistent with original approach
   ✓ No external methodology introduced

2. DATA DESCRIPTION:
   - Dataset: SelectOne task from objects.json
   - Categories: 16 object categories (fruit, vehicle, etc.)
   - Prompt templates: Referenced from original codebase
   ✓ Data description matches original repo structure

3. NUMERICAL CLAIMS:
   - GPT-2 results are from actual replication experiment
   - Original paper metrics (0.836-0.863, 8.26 ± 3.35) match documentation.pdf
   - No invented or hallucinated numbers detected
   ✓ All numbers are traceable

4. COMPARISON TABLE:
   - Original (Llama-70B) values: Verified against documentation.pdf
   - Replication (GPT-2) values: From actual experiments
   ✓ No fabricated comparisons

5. ANALYSIS CLAIMS:
   - Reasons for differences (model size, training, architecture): Reasonable
   - "GPT-2 not trained for instruction following": Factually correct
   - No unsupported speculative claims
   ✓ Analysis grounded in facts

6. REFERENCES TO ORIGINAL WORK:
   - Correctly cites it as investigating "filter heads"
   - Correctly describes the paper's hypothesis
   - No misrepresentation of original claims
   ✓ Original work described accurately

VERDICT: The replicated documentation does not introduce external references, 
invented findings, or hallucinated details. All claims are either:
(a) Verified against the original documentation
(b) Results from actual replication experiments
(c) Reasonable explanations for observed differences

DE3 RESULT: PASS
"""
print(de3_analysis)


=== DE3: NO EXTERNAL OR HALLUCINATED INFORMATION ===

CHECKING FOR EXTERNAL/HALLUCINATED CONTENT IN REPLICATION:

1. METHODOLOGY DESCRIPTION:
   - Query state caching: Matches original (Section 2.2)
   - Query state patching: Matches original methodology
   - Effect measurement: Consistent with original approach
   ✓ No external methodology introduced

2. DATA DESCRIPTION:
   - Dataset: SelectOne task from objects.json
   - Categories: 16 object categories (fruit, vehicle, etc.)
   - Prompt templates: Referenced from original codebase
   ✓ Data description matches original repo structure

3. NUMERICAL CLAIMS:
   - GPT-2 results are from actual replication experiment
   - Original paper metrics (0.836-0.863, 8.26 ± 3.35) match documentation.pdf
   - No invented or hallucinated numbers detected
   ✓ All numbers are traceable

4. COMPARISON TABLE:
   - Original (Llama-70B) values: Verified against documentation.pdf
   - Replication (GPT-2) values: From actual experiments
   ✓ No fabricat

In [17]:
# Create the output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Created output directory: {OUTPUT_DIR}")

Created output directory: /net/scratch2/smallyan/filter_eval/evaluation/replication_eval


In [18]:
# Create the documentation_evaluation_summary.md file
evaluation_summary_md = """# Documentation Evaluation Summary

## Results Comparison

The replicated documentation reports results using **GPT-2 (124M parameters)** while the original documentation presents findings from **Llama-3.3-70B-Instruct (70B parameters)** — a 560x difference in model size. This fundamental mismatch leads to significant discrepancies:

| Metric | Original (Llama-70B) | Replication (GPT-2) |
|--------|---------------------|---------------------|
| Causality Score | 0.836-0.863 | N/A (cannot compute) |
| ΔLogit | +9.03 ± 3.35 | +0.90 ± 1.43 |
| Task Accuracy | ~100% | <20% |

The replication shows approximately 10x smaller effect magnitudes and cannot compute the causality metric because GPT-2 fails to solve the underlying SelectOne task reliably. While the replication correctly implements the methodology, the results do not match the original within acceptable tolerance due to the model choice.

## Conclusions Comparison

The **original documentation** concludes that:
1. Filter heads are specialized attention heads localized in middle layers
2. Predicate representations are portable across formats, languages, and tasks
3. A dual filtering mechanism exists (lazy via filter heads vs eager via is_match flags)

The **replicated documentation** concludes that:
1. The methodology was correctly implemented
2. Some heads show stronger effects than others
3. Clear "filter heads" **cannot be identified** in GPT-2
4. Effect magnitude is much smaller than reported in the original

The replication explicitly states it could **not replicate** the core findings due to model limitations. While it doesn't contradict the original claims (attributing differences to model choice), it fails to confirm or support the key conclusions.

## External or Hallucinated Information

**No external or hallucinated information was detected.** All claims in the replicated documentation are:
- Verified against the original documentation (PDF)
- Derived from actual replication experiments
- Supported by reasonable explanations for observed differences

The replication accurately represents the original paper's claims and honestly reports its own experimental findings.

## Evaluation Checklist

| Criterion | Verdict | Notes |
|-----------|---------|-------|
| **DE1: Result Fidelity** | FAIL | Results differ by ~10x due to model mismatch (GPT-2 vs Llama-70B) |
| **DE2: Conclusion Consistency** | FAIL | Core conclusions could not be replicated; filter heads not identifiable in GPT-2 |
| **DE3: No External Information** | PASS | No hallucinated or external content introduced |

## Final Verdict

**REVISION REQUIRED**

The replication fails on DE1 (Result Fidelity) and DE2 (Conclusion Consistency) due to using an inadequate model (GPT-2 instead of Llama-70B). To achieve a passing grade, the replication should:

1. Use the same or comparable model (Llama-3.3-70B-Instruct or Gemma-2-27B-IT as mentioned in the original)
2. Demonstrate causality scores within the 0.8-0.9 range for SelectOne tasks
3. Confirm filter head localization in middle layers
4. Validate portability of predicate representations across linguistic variations
"""

# Write the file
with open(f"{OUTPUT_DIR}/documentation_evaluation_summary.md", 'w') as f:
    f.write(evaluation_summary_md)

print(f"Written: {OUTPUT_DIR}/documentation_evaluation_summary.md")
print("\n--- Content Preview ---")
print(evaluation_summary_md[:2000])

Written: /net/scratch2/smallyan/filter_eval/evaluation/replication_eval/documentation_evaluation_summary.md

--- Content Preview ---
# Documentation Evaluation Summary

## Results Comparison

The replicated documentation reports results using **GPT-2 (124M parameters)** while the original documentation presents findings from **Llama-3.3-70B-Instruct (70B parameters)** — a 560x difference in model size. This fundamental mismatch leads to significant discrepancies:

| Metric | Original (Llama-70B) | Replication (GPT-2) |
|--------|---------------------|---------------------|
| Causality Score | 0.836-0.863 | N/A (cannot compute) |
| ΔLogit | +9.03 ± 3.35 | +0.90 ± 1.43 |
| Task Accuracy | ~100% | <20% |

The replication shows approximately 10x smaller effect magnitudes and cannot compute the causality metric because GPT-2 fails to solve the underlying SelectOne task reliably. While the replication correctly implements the methodology, the results do not match the original within acceptab

In [19]:
import json

# Create the documentation_eval_summary.json file
evaluation_summary_json = {
    "Checklist": {
        "DE1_ResultFidelity": "FAIL",
        "DE2_ConclusionConsistency": "FAIL",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "The replication used GPT-2 (124M) instead of Llama-3.3-70B (70B), resulting in ~10x smaller effect magnitudes (ΔLogit 0.90 vs 9.03), inability to compute causality scores (task accuracy <20% vs ~100%), and failure to identify clear filter heads. Results do not match within acceptable tolerance.",
        "DE2_ConclusionConsistency": "The replication explicitly states that filter head localization could NOT be replicated with GPT-2, and the core conclusions about portable predicate representations and middle-layer localization cannot be validated. While the replication doesn't contradict the original, it fails to confirm any key findings.",
        "DE3_NoExternalInformation": "All claims in the replicated documentation are traceable: methodology descriptions match the original paper, numerical claims are from actual experiments, and comparison values are verified against the original documentation.pdf. No external references, invented findings, or hallucinated details were introduced."
    }
}

# Write the JSON file
with open(f"{OUTPUT_DIR}/documentation_eval_summary.json", 'w') as f:
    json.dump(evaluation_summary_json, f, indent=2)

print(f"Written: {OUTPUT_DIR}/documentation_eval_summary.json")
print("\n--- JSON Content ---")
print(json.dumps(evaluation_summary_json, indent=2))

Written: /net/scratch2/smallyan/filter_eval/evaluation/replication_eval/documentation_eval_summary.json

--- JSON Content ---
{
  "Checklist": {
    "DE1_ResultFidelity": "FAIL",
    "DE2_ConclusionConsistency": "FAIL",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "The replication used GPT-2 (124M) instead of Llama-3.3-70B (70B), resulting in ~10x smaller effect magnitudes (\u0394Logit 0.90 vs 9.03), inability to compute causality scores (task accuracy <20% vs ~100%), and failure to identify clear filter heads. Results do not match within acceptable tolerance.",
    "DE2_ConclusionConsistency": "The replication explicitly states that filter head localization could NOT be replicated with GPT-2, and the core conclusions about portable predicate representations and middle-layer localization cannot be validated. While the replication doesn't contradict the original, it fails to confirm any key findings.",
    "DE3_NoExternalInformation": "All clai

In [20]:
# Verify the output files exist
print("=== VERIFICATION ===")
print(f"\nOutput directory: {OUTPUT_DIR}")
print("\nFiles created:")
for f in os.listdir(OUTPUT_DIR):
    filepath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(filepath)
    print(f"  - {f} ({size} bytes)")

=== VERIFICATION ===

Output directory: /net/scratch2/smallyan/filter_eval/evaluation/replication_eval

Files created:
  - documentation_eval_summary.json (1186 bytes)
  - documentation_evaluation_summary.md (3144 bytes)


## Final Summary

### Documentation Evaluation Complete

| Criterion | Verdict |
|-----------|---------|
| **DE1: Result Fidelity** | FAIL |
| **DE2: Conclusion Consistency** | FAIL |
| **DE3: No External Information** | PASS |

### Final Verdict: **REVISION REQUIRED**

The replication documentation fails to match the original results and conclusions due to using GPT-2 (124M) instead of the original Llama-3.3-70B (70B). While the methodology was correctly implemented and no hallucinated information was introduced, the core findings could not be reproduced.

### Output Files
- `documentation_evaluation_summary.md` - Human-readable evaluation report
- `documentation_eval_summary.json` - Machine-readable checklist and rationale

In [21]:
# Final verification - read both files to confirm content
print("=== FINAL VERIFICATION ===\n")

print("1. documentation_evaluation_summary.md:")
print("-" * 50)
with open(f"{OUTPUT_DIR}/documentation_evaluation_summary.md", 'r') as f:
    print(f.read())

print("\n\n2. documentation_eval_summary.json:")
print("-" * 50)
with open(f"{OUTPUT_DIR}/documentation_eval_summary.json", 'r') as f:
    print(f.read())

=== FINAL VERIFICATION ===

1. documentation_evaluation_summary.md:
--------------------------------------------------
# Documentation Evaluation Summary

## Results Comparison

The replicated documentation reports results using **GPT-2 (124M parameters)** while the original documentation presents findings from **Llama-3.3-70B-Instruct (70B parameters)** — a 560x difference in model size. This fundamental mismatch leads to significant discrepancies:

| Metric | Original (Llama-70B) | Replication (GPT-2) |
|--------|---------------------|---------------------|
| Causality Score | 0.836-0.863 | N/A (cannot compute) |
| ΔLogit | +9.03 ± 3.35 | +0.90 ± 1.43 |
| Task Accuracy | ~100% | <20% |

The replication shows approximately 10x smaller effect magnitudes and cannot compute the causality metric because GPT-2 fails to solve the underlying SelectOne task reliably. While the replication correctly implements the methodology, the results do not match the original within acceptable tolerance d